### What is AUTOLOADER
_Databricks Auto Loader is a feature designed for efficient and incremental data ingestion from cloud object storage into Databricks. It is particularly well-suited for streaming data workloads, where new files continuously arrive in a specified cloud storage location_

Creating the dedicated volume to work on Autoloader

In [0]:
%sql
-- CREATE VOLUME databricksgovi.bronze.autoload_vol

## **AUTOLOADER QUERY**

**Initialization**

In [0]:
#AUTOLOADER INTIALIZATION
df = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format","csv")\
    .option("cloudFiles.schemaLocation", "/Volumes/databricksgovi/bronze/autoload_vol/destination/checkpoint/")\
    .option("cloudFiles.schemaEvolutionMode", "rescue")\
    .load("/Volumes/databricksgovi/bronze/autoload_vol/raw/")

**Starting the Stream**

In [0]:
df.writeStream.format("delta")\
        .outputMode("append")\
        .option("checkpointLocation", "/Volumes/databricksgovi/bronze/autoload_vol/destination/checkpoint/")\
        .trigger(once=True)\
        .start("/Volumes/databricksgovi/bronze/autoload_vol/destination/data/")

In [0]:
df = spark.read.format("delta")\
        .load("/Volumes/databricksgovi/bronze/autoload_vol/destination/data")
display(df)

### **Fetching the Rescued columns**

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
rescued_schema = StructType()\
    .add("discount", StringType())\
    .add("payment_method", StringType())

#parse the stringified JSON column
df = df.withColumn("rescued_struct", from_json(col("_rescued_data"), rescued_schema))

#Extract individual fields
df = df.withColumn("rescued_discount", col("rescued_struct.discount"))\
    .withColumn("rescued_payment_method", col("rescued_struct.payment_method"))

In [0]:
display(df)